# Lesson 14A: Offline RL and Imitation - Theory

## Introduction

Every algorithm so far assumed the agent could keep interacting with the environment to correct its own mistakes. **Offline (batch) RL** removes that assumption entirely: learn a policy from a **fixed dataset** of transitions, with no further environment interaction at all — the setting whenever online exploration is expensive, slow, or dangerous (healthcare, robotics, autonomous driving).

- **Behavioral cloning**: the simplest approach — supervised learning on expert demonstrations
- **Distribution shift**: why BC's errors compound instead of staying bounded
- **Inverse RL and GAIL**: learn the reward function (or an implicit substitute for it) from demonstrations, rather than just the action mapping
- **Conservative Q-Learning (CQL)**: the fix for offline value-based RL's core failure mode — unconstrained overestimation of actions the dataset never tried

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

## Part 1: Motivating Offline RL

Online RL alternates acting and learning: try something, observe the outcome, correct. Offline RL only gets a fixed dataset $\mathcal{D} = \{(s, a, r, s')\}$, collected once by some other (possibly unknown, possibly suboptimal) **behavior policy** $\pi_\beta$, with zero further environment access during training.

This isn't a minor variant of online RL — it changes what's achievable. A policy can only be evaluated at deployment, after training is complete, so there's no way to detect and correct a bad extrapolation the way an online agent naturally would on its next environment interaction. The entire offline RL and imitation-learning literature is, in one sense, a series of answers to a single question: **how do you avoid confidently extrapolating into territory the dataset never covered?**

## Part 2: Behavioral Cloning and Distribution Shift

**Behavioral cloning (BC)** is supervised learning applied directly to demonstrations: fit $\pi_\theta(a \mid s)$ to minimize $\mathbb{E}_{(s,a) \sim \mathcal{D}}[-\log \pi_\theta(a \mid s)]$, exactly like training a classifier. No reward signal needed at all — just (state, expert-action) pairs.

The problem is **distribution shift** (Ross & Bagnell, 2010): BC minimizes error on states the *expert* visits, but once deployed, the cloned policy's own small errors push it into states the expert rarely or never visited — states BC was never trained on. Nothing in the supervised-learning objective bounds behavior off the training distribution, so a small per-step error rate compounds: each mistake shifts the state distribution further from what BC knows, producing more mistakes, in a feedback loop the expert's own on-policy data collection would have caught and corrected.

In [ ]:
from stable_baselines3 import PPO

env = gym.make('CartPole-v1')

# Train an expert (SB3 PPO): this "generates" the demonstrations BC will clone.
expert = PPO('MlpPolicy', env, verbose=0, device='cpu')
expert.learn(total_timesteps=30000)


def evaluate(action_fn, n_episodes=20):
    lengths = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=500 + ep)
        for t in range(500):
            a = action_fn(s)
            s, r, terminated, truncated, _ = env.step(a)
            if terminated or truncated:
                break
        lengths.append(t + 1)
    return lengths


expert_lengths = evaluate(lambda s: expert.predict(s, deterministic=True)[0])
print(f"Expert mean episode length: {np.mean(expert_lengths):.0f} / 500")


def collect_demonstrations(n_episodes=50):
    states, actions = [], []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        for _ in range(500):
            a, _ = expert.predict(s, deterministic=True)
            states.append(s)
            actions.append(int(a))
            s, r, terminated, truncated, _ = env.step(a)
            if terminated or truncated:
                break
    return np.array(states), np.array(actions)


demo_states, demo_actions = collect_demonstrations()
print(f"Collected {len(demo_states)} demonstration transitions from {50} expert episodes")


class BehavioralCloningPolicy(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))

    def forward(self, x):
        return self.net(x)


bc_policy = BehavioralCloningPolicy()
optimizer = optim.Adam(bc_policy.parameters(), lr=1e-3)
X = torch.as_tensor(demo_states, dtype=torch.float32)
Y = torch.as_tensor(demo_actions, dtype=torch.long)
for epoch in range(100):
    logits = bc_policy(X)
    loss = nn.functional.cross_entropy(logits, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"BC training loss (pure supervised classification, no reward used): {loss.item():.3f}")


def bc_action(s):
    with torch.no_grad():
        return int(torch.argmax(bc_policy(torch.as_tensor(s, dtype=torch.float32).unsqueeze(0))).item())


bc_lengths = evaluate(bc_action)
print(f"BC mean episode length:     {np.mean(bc_lengths):.0f} / 500")
print("\nThe expert solves CartPole outright (500/500 every episode). BC, trained on the exact")
print("same demonstrations via ordinary supervised learning, degrades sharply -- distribution")
print("shift in action: BC's own small errors push it into states the expert rarely visited,")
print("where BC was never trained and has no reason to generalize correctly.")

## Part 3: Inverse RL and GAIL

### Inverse Reinforcement Learning (IRL)

BC clones the *mapping* from states to actions, throwing away the *reason* those actions were chosen. IRL instead infers the **reward function** the expert appears to be optimizing, then runs ordinary RL against that recovered reward. Maximum-entropy IRL (Ziebart et al., 2008) frames this probabilistically: assume the expert's trajectory distribution is proportional to $\exp(R(\tau))$ and find the reward parameters that make the observed demonstrations most likely.

Recovering an explicit reward has a real advantage over BC: a reward function is a much more *compact*, transferable description of the task than a full state-to-action mapping, and once recovered it can be optimized in environment conditions the demonstrations never covered.

### GAIL (Generative Adversarial Imitation Learning)

GAIL (Ho & Ermon, 2016) sidesteps the expensive inner RL loop that classic IRL needs by casting imitation as a **GAN**:

- A **discriminator** $D_\phi(s, a)$ is trained to distinguish expert $(s,a)$ pairs from policy-generated $(s,a)$ pairs
- The **policy** $\pi_\theta$ is trained via RL to *fool* the discriminator, using $-\log D_\phi(s, a)$ as a reward signal — states and actions the discriminator can't distinguish from the expert's get high implicit reward

$$\min_\theta \max_\phi \; \mathbb{E}_{\pi_\theta}[\log D_\phi(s,a)] + \mathbb{E}_{\pi_E}[\log(1 - D_\phi(s,a))]$$

The discriminator never hands over an explicit, reusable reward function the way IRL does — but training is dramatically cheaper, and unlike BC, GAIL's policy generates its **own** on-policy rollouts during training, so it directly experiences and corrects for exactly the kind of distribution shift that sinks pure BC.

## Part 4: Conservative Q-Learning (CQL)

Value-based offline RL has its own, sharper version of the distribution-shift problem. Bellman backups bootstrap through $\max_a Q(s', a)$ — and with a fixed dataset, nothing stops that max from landing on an action the dataset never actually tried at $s'$. Function approximators extrapolate confidently (and often wrongly) into unseen state-action regions, so a rarely-tried action's Q-value can drift arbitrarily high through repeated bootstrapping, with **no environment interaction ever available to correct it**. Standard Q-learning, applied offline, systematically overestimates.

CQL (Kumar et al., 2020) adds a regularizer that directly pushes down Q-values for actions **not supported by the dataset**, while pushing up Q-values for the actions the dataset actually took:

$$\mathcal{L}_{\text{CQL}} = \alpha \cdot \mathbb{E}_{s \sim \mathcal{D}}\left[\log \sum_a \exp Q(s, a) - \mathbb{E}_{a \sim \mathcal{D}}[Q(s, a)]\right] + \mathcal{L}_{\text{Bellman}}$$

The log-sum-exp term is a soft maximum over *all* actions (pushing every action's Q down), while the second term pulls the *dataset's* actions back up — net effect: a systematic downward bias specifically on actions the data doesn't support. This trades a small amount of pessimism for a large reduction in offline overestimation error — the accepted trade in a setting with no way to online-correct a mistake.

In [ ]:
SIZE = 6
GOAL = (SIZE - 1, SIZE - 1)
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def maze_step(pos, action):
    if pos == GOAL:
        return pos, 0.0, True
    di, dj = DELTAS[action]
    ni, nj = pos[0] + di, pos[1] + dj
    if 0 <= ni < SIZE and 0 <= nj < SIZE:
        pos = (ni, nj)
    done = pos == GOAL
    reward = 10.0 if done else -1.0
    return pos, reward, done


def collect_offline_dataset(n_episodes=100, max_steps=15):
    """A NARROW behavior policy: mostly moves down/right, rarely up/left --
    so up/left is under-represented almost everywhere in the resulting dataset."""
    dataset = []
    for _ in range(n_episodes):
        pos = (0, 0)
        for _ in range(max_steps):
            action = np.random.choice([0, 1, 2, 3], p=[0.05, 0.55, 0.05, 0.35])
            next_pos, reward, done = maze_step(pos, action)
            dataset.append((pos, action, reward, next_pos, done))
            pos = next_pos
            if done:
                break
    return dataset


offline_dataset = collect_offline_dataset()


def encode(state):
    return np.array([state[0] / (SIZE - 1), state[1] / (SIZE - 1)], dtype=np.float32)


class OfflineQNet(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 4))

    def forward(self, x):
        return self.net(x)


def train_offline(use_cql, n_epochs=100, batch_size=64, gamma=0.95, lr=1e-3, cql_alpha=1.0):
    net = OfflineQNet()
    target_net = OfflineQNet()
    target_net.load_state_dict(net.state_dict())
    optimizer = optim.Adam(net.parameters(), lr=lr)

    states = torch.as_tensor(np.array([encode(d[0]) for d in offline_dataset]))
    actions = torch.as_tensor([d[1] for d in offline_dataset], dtype=torch.long)
    rewards = torch.as_tensor([d[2] for d in offline_dataset], dtype=torch.float32)
    next_states = torch.as_tensor(np.array([encode(d[3]) for d in offline_dataset]))
    dones = torch.as_tensor([float(d[4]) for d in offline_dataset])
    n = len(offline_dataset)

    for epoch in range(n_epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            q_all = net(states[idx])
            q_taken = q_all.gather(1, actions[idx].unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                target = rewards[idx] + gamma * (1 - dones[idx]) * torch.max(target_net(next_states[idx]), dim=1).values
            bellman_loss = nn.functional.mse_loss(q_taken, target)

            loss = bellman_loss
            if use_cql:
                # The real CQL regularizer: log-sum-exp over ALL actions (soft max, pushes
                # every action down) minus the Q of the action the DATASET actually took
                # (pulls that one back up) -- net effect: push down whatever the data didn't see.
                logsumexp = torch.logsumexp(q_all, dim=1)
                cql_penalty = (logsumexp - q_taken).mean()
                loss = bellman_loss + cql_alpha * cql_penalty

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 20 == 0:
            target_net.load_state_dict(net.state_dict())
    return net


naive_net = train_offline(use_cql=False)
cql_net = train_offline(use_cql=True)

all_states = [(i, j) for i in range(SIZE) for j in range(SIZE)]


def mean_q(net):
    with torch.no_grad():
        return np.mean([net(torch.as_tensor(encode(s)).unsqueeze(0)).numpy() for s in all_states])


print(f"Naive offline Q-learning -- mean Q-value across the grid: {mean_q(naive_net):.2f}")
print(f"CQL offline Q-learning   -- mean Q-value across the grid: {mean_q(cql_net):.2f}")
print("\nSame Bellman backups, same fixed dataset -- the CQL regularizer alone produces")
print("systematically lower (more conservative) value estimates, exactly the pessimism the")
print("theory predicts: better to underestimate than to confidently overestimate an action")
print("nothing in the data ever backs up.")

## Part 5: Learning from Demonstrations

Putting the pieces together, three answers to "how do I use demonstrations?", in increasing order of what they need:

| Method | Needs reward labels? | Needs environment access during training? | Handles distribution shift? |
|---|---|---|---|
| Behavioral cloning | No | No | No — this is exactly its failure mode |
| IRL | No (recovers one) | Yes (inner RL loop on recovered reward) | Better — reward-driven, not action-mimicking |
| GAIL | No | Yes (on-policy rollouts against the discriminator) | Yes — on-policy data collection corrects it directly |
| Offline RL + CQL | Yes (in the dataset) | **No** | Yes — via explicit pessimism about OOD actions |

The practical choice depends entirely on what's available: demonstrations only (no reward, no further environment access) points to BC or GAIL; a reward-labeled but frozen dataset (no environment access at all) points to offline RL with a conservative correction like CQL.

## Key Concepts

1. **Offline RL**: learn entirely from a fixed dataset, zero further environment interaction — common whenever online exploration is costly or dangerous
2. **Distribution shift**: BC's fundamental failure mode — supervised learning has no mechanism to bound behavior off the training distribution, so small errors compound
3. **IRL**: recover the reward function the expert appears to optimize, rather than cloning its action mapping directly
4. **GAIL**: casts imitation as a GAN — a discriminator's confusion becomes the policy's reward, avoiding both BC's distribution shift (GAIL collects its own on-policy data) and classic IRL's expensive inner RL loop
5. **CQL**: penalizes Q-values for actions the dataset doesn't support via a log-sum-exp regularizer, directly countering offline value-based RL's core failure — unconstrained overestimation through bootstrapping with no environment access to correct it